In [1]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

num_vals = 3

A = model.NewIntVar(0, num_vals - 1, "A")
B = model.NewIntVar(0, num_vals - 1, "B")
C = model.NewIntVar(0, num_vals - 1, "C")
D = model.NewIntVar(0, num_vals - 1, "D")
E = model.NewIntVar(0, num_vals - 1, "E")

model.add(A != B)
model.add(A != E)
model.add(B != C)
model.add(B != D)
model.add(C != D)
model.add(D != E)

class SolutionPrinter(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.variables = variables
        self.solution_count = 0

    def on_solution_callback(self):
        self.solution_count += 1
        for v in self.variables:
            print(f"{v}={self.value(v)}", end=" ")
        print()

solver = cp_model.CpSolver()
solution_printer = SolutionPrinter([A, B, C, D, E])

solver.parameters.enumerate_all_solutions = True
solver.solve(model, solution_printer)

print("Total solutions:", solution_printer.solution_count)


ModuleNotFoundError: No module named 'ortools'

In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

num_items = 10

Mon = model.new_int_var(0, num_items - 1, "Mon")
Tue = model.new_int_var(0, num_items - 1, "Tue")
Wed = model.new_int_var(0, num_items - 1, "Wed")
Thu = model.new_int_var(0, num_items - 1, "Thu")
Fri = model.new_int_var(0, num_items - 1, "Fri")

model.add(Fri <= 1)

model.add(Mon >= 2)
model.add(Thu >= 2)

model.add_all_different([Mon, Tue, Wed, Thu, Fri])

class SolutionPrinter(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.variables = variables
        self.solution_count = 0

    def on_solution_callback(self):
        self.solution_count += 1
        print(f"\nSolution {self.solution_count}:")
        for v in self.variables:
            print(f"{v}={self.value(v)}", end=" ")
        print()

solver = cp_model.CpSolver()
solution_printer = SolutionPrinter([Mon, Tue, Wed, Thu, Fri])

solver.parameters.enumerate_all_solutions = True
solver.solve(model, solution_printer)

print("\n0=SQ1, 1=SQ2, 2-6=Shirts, 7-9=Pants")
print("\nTotal solutions:", solution_printer.solution_count)


In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

n = 6

grid = []
for i in range(n):
    row = []
    for j in range(n):
        row.append(model.new_int_var(1, 6, f"cell_{i}_{j}"))
    grid.append(row)

puzzle = [
    [0, 0, 6, 2, 5, 0],
    [0, 0, 0, 4, 6, 0],
    [0, 1, 2, 0, 0, 0],
    [5, 6, 0, 0, 0, 4],
    [0, 0, 4, 3, 0, 2],
    [3, 0, 0, 5, 0, 6]
]

for i in range(n):
    for j in range(n):
        if puzzle[i][j] != 0:
            model.add(grid[i][j] == puzzle[i][j])

for i in range(n):
    model.add_all_different(grid[i])

for j in range(n):
    model.add_all_different([grid[i][j] for i in range(n)])

for i in range(0, n, 2):
    for j in range(0, n, 3):
        block = []
        for di in range(2):
            for dj in range(3):
                block.append(grid[i + di][j + dj])
        model.add_all_different(block)

solver = cp_model.CpSolver()
status = solver.solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("Solved Sudoku:\n")
    for i in range(n):
        for j in range(n):
            print(solver.value(grid[i][j]), end=" ")
        print()
else:
    print("No solution found")
